In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
from enum import Enum
class Extension(Enum):
    GZ = '.gz'
    TAR_GZ = '.tar.gz'
    BZ2 = '.bz2'
    H5 = '.h5'
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- radar_index_radolan ---
FIX_RADAR_INDEX_RADOLAN_PD = pd.DataFrame({
    "filename": ["/bin/a.gz", "/bin/b.tar.gz", "/bin/c.bz2", "/other/d.gz"],
    "datetime": ["2020-01-01", "2020-01-02", "2020-01-03", "2020-01-04"],
})
FIX_RADAR_INDEX_RADOLAN_PL = pl.from_pandas(FIX_RADAR_INDEX_RADOLAN_PD)

# --- radar_index_sweeps ---
FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PD = pd.DataFrame({"filename": ["sweep_a.bz2", "sweep_b.bz2"], "datetime": ["2020-01-01", "2020-01-02"]})
FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PL = pl.from_pandas(FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PD)
FIX_RADAR_INDEX_SWEEPS_MIXED_PD = pd.DataFrame({"filename": ["sweep_a.bz2", "sweep_b.txt"], "datetime": ["2020-01-01", "2020-01-02"]})
FIX_RADAR_INDEX_SWEEPS_MIXED_PL = pl.from_pandas(FIX_RADAR_INDEX_SWEEPS_MIXED_PD)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_radar_index_radolan(file_index=None):
    if file_index is None:
        file_index = pd.DataFrame({"filename":["abc.tar.gz"],"datetime":["2020-01-01"]})
    file_index = file_index[
        file_index["filename"].str.contains("/bin/")
        & file_index["filename"].str.endswith((Extension.GZ.value, Extension.TAR_GZ.value))
    ].copy()
    return file_index

def before_radar_index_sweeps(files_server=None):
    if files_server is None:
        files_server = pd.DataFrame({"filename":["abc.bz2"],"datetime":["2020-01-01"]})
    if not all(files_server["filename"].str.endswith(".bz2")):
        files_server = files_server[~files_server["filename"].str.endswith(".bz2")]
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_radar_index_radolan(file_index=None):
    if file_index is None:
        file_index = pl.DataFrame({"filename":["abc.tar.gz"],"datetime":["2020-01-01"]})

    file_index = file_index.filter(
        pl.col("filename").str.contains("/bin/")
        & (
            pl.col("filename").str.ends_with(Extension.GZ.value)
            | pl.col("filename").str.ends_with(Extension.TAR_GZ.value)
        )
    ).clone()
    return file_index

def gen_radar_index_sweeps(files_server=None):
    if files_server is None:
        files_server = pl.DataFrame({"filename":["abc.bz2"],"datetime":["2020-01-01"]})

    if not files_server["filename"].str.ends_with(".bz2").all():
        files_server = files_server.filter(~pl.col("filename").str.ends_with(".bz2"))
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:

# === Tests: radar_index_sweeps ===
import sys

def _capture_sweep_frame(func, frame):
    captured = {}
    code = func.__code__
    def _trace(call_frame, event, arg):
        if event == "return" and call_frame.f_code is code: captured.update(call_frame.f_locals)
        return _trace
    old = sys.gettrace(); sys.settrace(_trace)
    try: result = func(frame)
    finally: sys.settrace(old)
    return result, captured.get("files_server")

try:
    _r = gen_radar_index_sweeps(FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PL)
    print("✅ L1 smoke gen_radar_index_sweeps: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_radar_index_sweeps: {type(_e).__name__}: {_e}")

try:
    _rb = before_radar_index_sweeps(FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PD.copy())
    print("✅ L1 smoke before_radar_index_sweeps: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_radar_index_sweeps: {type(_e).__name__}: {_e}")

for _label, _pd_frame, _pl_frame in [
    ("all bz2", FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PD, FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PL),
    ("mixed suffixes", FIX_RADAR_INDEX_SWEEPS_MIXED_PD, FIX_RADAR_INDEX_SWEEPS_MIXED_PL),
]:
    try:
        _rb, _before_frame = _capture_sweep_frame(before_radar_index_sweeps, _pd_frame.copy())
        _rg, _gen_frame = _capture_sweep_frame(gen_radar_index_sweeps, _pl_frame)
        if _rb is not None or _rg is not None:
            print(f"❌ L2 equivalence radar_index_sweeps {_label}: MISMATCH — return contract")
        else:
            compare(_before_frame, _gen_frame, f"L2 equivalence radar_index_sweeps {_label}", check_row_order=True)
    except Exception as _e:
        print(f"❌ L2 equivalence radar_index_sweeps {_label}: {type(_e).__name__}: {_e}")

try:
    _empty_pd = FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PD.head(0)
    _empty_pl = FIX_RADAR_INDEX_SWEEPS_ALL_BZ2_PL.head(0)
    _rb, _before_frame = _capture_sweep_frame(before_radar_index_sweeps, _empty_pd)
    _rg, _gen_frame = _capture_sweep_frame(gen_radar_index_sweeps, _empty_pl)
    compare(_before_frame, _gen_frame, "L3 edge radar_index_sweeps empty", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge radar_index_sweeps empty: {type(_e).__name__}: {_e}")
